In [2]:
import torch
import torch.nn as nn
from torchinfo import summary
from sklearn.model_selection import train_test_split

In [3]:
X = torch.rand(10,5)
y = torch.tensor([1,0,1,0,1,0,1,0,1,0], dtype=torch.float32)

In [4]:
# sequential way of building a model
# we can also build a model using nn.Sequential, which is a container module that sequences together a series of modules.
# the nn.Sequential class allows us to build a model by simply stacking layers together in a sequential manner.
class nn_model(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.model = nn.Sequential(nn.Linear(num_features,3),
                                 nn.ReLU(),
                                 nn.Linear(3,1),
                                 nn.Sigmoid())

  def forward(self, x):
    x = self.model(x)
    return x

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

# initializing the model, loss function and optimizer
model = nn_model(num_features=X.shape[1])
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
epochs = 10

summary(model, input_size=(1,X.shape[1]))

Layer (type:depth-idx)                   Output Shape              Param #
nn_model                                 [1, 1]                    --
├─Sequential: 1-1                        [1, 1]                    --
│    └─Linear: 2-1                       [1, 3]                    18
│    └─ReLU: 2-2                         [1, 3]                    --
│    └─Linear: 2-3                       [1, 1]                    4
│    └─Sigmoid: 2-4                      [1, 1]                    --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [11]:
for epoch in range(epochs):

    # zero the gradients
    # before we perform the backward pass, we need to zero the gradients. 
    # this is because by default, gradients are accumulated in PyTorch. if we don't zero the gradients, 
    # the new gradients will be added to the existing gradients, which can lead to incorrect updates of the model parameters.
    optimizer.zero_grad()  

    # forward pass
    y_pred = model(X_train)
    # calculating the loss
    loss = loss_fn(y_pred, y_train.view(-1,1))
    #backward pass
    loss.backward()
    # update the weights
    optimizer.step()
   

    print(f'Epoch : {epoch+1} | Loss : {loss.item()}')

Epoch : 1 | Loss : 0.7291613221168518
Epoch : 2 | Loss : 0.7290695905685425
Epoch : 3 | Loss : 0.7289780974388123
Epoch : 4 | Loss : 0.7288867235183716
Epoch : 5 | Loss : 0.72879558801651
Epoch : 6 | Loss : 0.7287046313285828
Epoch : 7 | Loss : 0.7286138534545898
Epoch : 8 | Loss : 0.7285232543945312
Epoch : 9 | Loss : 0.728432834148407
Epoch : 10 | Loss : 0.7283425331115723


In [12]:
model.eval() 
# eval mode is used to set the model to evaluation mode, which is necessary when we are evaluating the 
# model on a test set or making predictions. it turns off certain layers like dropout and batch normalization 
# that behave differently during training and evaluation.

with torch.no_grad():
  y_pred = model(X_test)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test.view(-1,1)).float().mean()
  print(f'Accuracy : {accuracy}')

Accuracy : 0.5
